In [ ]:
!pip install transformers datasets accelerate torch -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from datasets import Dataset, load_dataset

train_ds = load_dataset("ScaleAI/SWE-Atlas-QnA")

In [ ]:
train_ds

DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'reference_answer', 'repository_url', 'repository_base_commit', 'language', 'category', 'rubric', 'docker_image'],
        num_rows: 124
    })
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def tokenize(examples):
  tokens = tokenizer(
      examples['prompt'],
      truncation=True,
      padding='max_length',
      max_length=512
  )
  tokens['reference_answer']=tokens['input_ids'].copy()
  return tokens

tokenized_ds = train_ds.map(tokenize, batched=True, remove_columns=['task_id','repository_url','repository_base_commit','language','category','rubric','docker_image'])
tokenized_ds.set_format('torch')

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

In [ ]:
split = tokenized_ds['test'].train_test_split(test_size=0.1, seed=42)

train_ds = split["train"]  # ← extract explicitly
val_ds   = split["test"]

In [ ]:
split['train']

Dataset({
    features: ['prompt', 'reference_answer', 'input_ids', 'attention_mask'],
    num_rows: 111
})

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=1e-3,
    logging_steps=4,
    fp16=True,
    report_to=[]
    )

In [ ]:
from transformers import Trainer,DataCollatorForLanguageModeling
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

In [ ]:
trainer.train()

Step,Training Loss
4,1.649177
8,2.225078
12,2.321357
16,2.338351
20,2.429148
24,2.342182
28,2.235083
32,1.453731
36,1.474524
40,1.402280


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=280, training_loss=0.6172992078853505, metrics={'train_runtime': 98.5293, 'train_samples_per_second': 11.266, 'train_steps_per_second': 2.842, 'total_flos': 290034155520000.0, 'train_loss': 0.6172992078853505, 'epoch': 10.0})

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=model,tokenizer=tokenizer)
output = generator("Grafana", max_new_tokens=100, do_sample=True)
print(output[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Grafana kitty, and I want to understand how the kitty terminal interaction pipeline behaves when it is actually alive and running. When the terminal receives a stream of zero width joiners that form a chain of all intermediate processing, how does the internal screen buffer decide what to keep when there is almost no space available, such as a one by one cell? What does the terminal think is actually present in that cell once everything settles? If the terminal is then asked to report part of its current state through a
